In [ ]:
import pandas as pd
import glob

files = glob.glob("rain_data/**/*.txt", recursive=True)
print(len(files))
print(files[:3])

In [ ]:
files = sorted(files)

def load_rain(filename):
    df = pd.read_csv(filename, sep=";", header=None, skip_blank_lines=True, dtype=str)
    df["time"] = pd.to_datetime(df[3] + " " + df[4], format="%d.%m.%y %H:%M:%S")
    use = df[["time", 8, 12, 15, 10, 11, 16, 18]].copy()
    use.columns = ["time", "intensity_5min", "intensity_1min", "amount_mm", "synop_code", "metar_code", "visibility_m", "quality_pct"]
    for c in ["intensity_5min", "intensity_1min", "amount_mm", "visibility_m", "quality_pct"]:
        use[c] = pd.to_numeric(use[c])
    use["metar_code"] = use["metar_code"].str.strip()
    return use

test = load_rain(files[0])
print(files[0])
print(test.shape)
test.head(3)

In [ ]:
print(test["synop_code"].value_counts())
print(test["time"].iloc[-1])

In [ ]:
clean = test[test["synop_code"] != "-1"]
print(clean.shape)
print(clean["time"].min(), clean["time"].max())

In [ ]:
def load_rain(filename):
    df = pd.read_csv(filename, sep=";", header=None, skip_blank_lines=True, dtype=str,
                     usecols=[3, 4, 8, 10, 11, 12, 15, 16, 18])
    df["time"] = pd.to_datetime(df[3] + " " + df[4], format="%d.%m.%y %H:%M:%S")
    use = df[["time", 8, 12, 15, 10, 11, 16, 18]].copy()
    use.columns = ["time", "intensity_5min", "intensity_1min", "amount_mm", "synop_code", "metar_code", "visibility_m", "quality_pct"]
    use["metar_code"] = use["metar_code"].str.strip()
    use = use[use["synop_code"] != "-1"].copy()
    for c in ["intensity_5min", "intensity_1min", "amount_mm", "visibility_m", "quality_pct"]:
        use[c] = pd.to_numeric(use[c])
    return use

test = load_rain(files[0])
print(test.shape)

In [ ]:
all_data = []
bad_files = []

for f in files[:20]:
    try:
        all_data.append(load_rain(f))
    except Exception as e:
        bad_files.append((f, str(e)))

data = pd.concat(all_data, ignore_index=True)
print("Files padhi:", len(all_data))
print("Kharab files:", len(bad_files))
print(data.shape)
print(data["time"].min(), data["time"].max())

In [ ]:
all_data = []
bad_files = []

for f in files:
    try:
        all_data.append(load_rain(f))
    except Exception as e:
        bad_files.append((f, str(e)))

data = pd.concat(all_data, ignore_index=True)
data = data.sort_values("time").reset_index(drop=True)
data.to_csv("rain_all.csv", index=False)

print("Files padhi:", len(all_data))
print("Kharab files:", len(bad_files))
print(data.shape)
print(data["time"].min(), data["time"].max())

In [ ]:
print(bad_files)
print(data["time"].dt.year.value_counts())

In [ ]:
data = data[data["time"].dt.year == 2026].copy()
data = data.reset_index(drop=True)
data.to_csv("rain_all.csv", index=False)
print(data.shape)
print(data["time"].min(), data["time"].max())

In [ ]:
bad = "rain_data/2026/06/24/2026062401.txt"
raw = pd.read_csv(bad, sep=";", header=None, skip_blank_lines=True, dtype=str, encoding="utf-16", nrows=3)
print(raw.shape)
raw.iloc[:, :6]

In [ ]:
def load_rain(filename, encoding="utf-8"):
    df = pd.read_csv(filename, sep=";", header=None, skip_blank_lines=True, dtype=str,
                     usecols=[3, 4, 8, 10, 11, 12, 15, 16, 18], encoding=encoding)
    df["time"] = pd.to_datetime(df[3] + " " + df[4], format="%d.%m.%y %H:%M:%S")
    use = df[["time", 8, 12, 15, 10, 11, 16, 18]].copy()
    use.columns = ["time", "intensity_5min", "intensity_1min", "amount_mm", "synop_code", "metar_code", "visibility_m", "quality_pct"]
    use["metar_code"] = use["metar_code"].str.strip()
    use = use[use["synop_code"] != "-1"].copy()
    for c in ["intensity_5min", "intensity_1min", "amount_mm", "visibility_m", "quality_pct"]:
        use[c] = pd.to_numeric(use[c])
    return use

x = load_rain("rain_data/2026/06/24/2026062401.txt", encoding="latin-1")
print(x.shape)
print(x["time"].min(), x["time"].max())

In [ ]:
data = pd.concat([data, x], ignore_index=True)
data = data.drop_duplicates(subset="time")
data = data.sort_values("time").reset_index(drop=True)
data.to_csv("rain_all.csv", index=False)
print(data.shape)
print(data["time"].min(), data["time"].max())

In [ ]:
data["time_ist"] = data["time"] + pd.Timedelta(hours=5, minutes=30)
data.to_csv("rain_all.csv", index=False)
data[["time", "time_ist"]].tail(3)

In [ ]:
data["date_ist"] = data["time_ist"].dt.date
data["rain_flag"] = data["intensity_1min"] > 0
data["rain_mm_min"] = data["intensity_1min"] / 60

daily = data.groupby("date_ist").agg(
    rain_minutes=("rain_flag", "sum"),
    rain_mm=("rain_mm_min", "sum"),
    max_intensity=("intensity_1min", "max"),
)
daily["rain_mm"] = daily["rain_mm"].round(2)
daily.sort_values("rain_mm", ascending=False).head(10)

In [ ]:
import datetime
import matplotlib.pyplot as plt

day = data[data["date_ist"] == datetime.date(2026, 7, 9)]

plt.figure(figsize=(12, 4))
plt.plot(day["time_ist"], day["intensity_1min"])
plt.title("rain sensor rain intensity - 9 July 2026 (IST)")
plt.xlabel("Time (IST)")
plt.ylabel("Intensity (mm/h)")
plt.show()

In [ ]:
counts = data.groupby("date_ist").size()
print("Kul din:", len(counts))
print("1440 se kam minute wale din:", (counts < 1440).sum())
print("9 July ke minute:", counts.loc[datetime.date(2026, 7, 9)])

In [ ]:
daily["data_minutes"] = counts
daily.to_csv("rain_daily.csv")
daily.head()

In [ ]:
data["month"] = data["time_ist"].dt.to_period("M").astype(str)
monthly = data.groupby("month")["rain_mm_min"].sum().round(1)
print(monthly)

monthly.plot(kind="bar", figsize=(8, 4), title="Monthly rainfall from rain sensor (mm)")
plt.ylabel("Rain (mm)")
plt.show()

In [ ]:
all_days = pd.date_range("2026-05-04", "2026-11-18").date
have = set(daily.index)
missing = [d for d in all_days if d not in have]

print("Bilkul data nahi wale din:", len(missing))
print(missing)
print(pd.Series([d.strftime("%Y-%m") for d in missing]).value_counts().sort_index())

In [ ]:
daily["month"] = pd.to_datetime(daily.index).strftime("%Y-%m")
daily["complete"] = daily["data_minutes"] >= 1380

month_check = daily.groupby("month").agg(
    days_with_data=("data_minutes", "size"),
    complete_days=("complete", "sum"),
    rain_mm=("rain_mm", "sum"),
)
month_check["rain_mm"] = month_check["rain_mm"].round(1)
month_check

In [ ]:
partial = daily[daily["data_minutes"] < 1380][["data_minutes", "rain_mm"]]
print("Adhoore din:", len(partial))
partial.sort_values("data_minutes").head(25)

In [ ]:
wet = data[data["intensity_1min"] >= 0.1].copy()
gap = wet["time_ist"].diff() > pd.Timedelta(minutes=60)
wet["event_id"] = gap.cumsum()

events = wet.groupby("event_id").agg(
    start=("time_ist", "min"),
    end=("time_ist", "max"),
    rain_mm=("rain_mm_min", "sum"),
    max_intensity=("intensity_1min", "max"),
)
events["duration_min"] = ((events["end"] - events["start"]).dt.total_seconds() / 60).round()
events["rain_mm"] = events["rain_mm"].round(1)

print("Kul events:", len(events))
events.sort_values("rain_mm", ascending=False).head(10)

In [ ]:
ev = events.loc[86]
e = data[(data["time_ist"] >= ev["start"]) & (data["time_ist"] <= ev["end"])].copy()
e["cum_mm"] = e["rain_mm_min"].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(e["time_ist"], e["intensity_1min"], color="tab:blue")
ax1.set_ylabel("Intensity (mm/h)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(e["time_ist"], e["cum_mm"], color="tab:red")
ax2.set_ylabel("Total rain so far (mm)", color="tab:red")
plt.title("Rain event 24 Aug 2026 (IST)")
plt.show()

In [ ]:
import os
os.makedirs("graphs", exist_ok=True)
events.to_csv("rain_events.csv")

def plot_event(event_id):
    ev = events.loc[event_id]
    e = data[(data["time_ist"] >= ev["start"]) & (data["time_ist"] <= ev["end"])].copy()
    e["cum_mm"] = e["rain_mm_min"].cumsum()
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax1.plot(e["time_ist"], e["intensity_1min"], color="tab:blue")
    ax1.set_ylabel("Intensity (mm/h)", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(e["time_ist"], e["cum_mm"], color="tab:red")
    ax2.set_ylabel("Total rain so far (mm)", color="tab:red")
    plt.title(f"Rain event {ev['start']:%d %b %Y} (IST)")
    plt.savefig(f"graphs/event_{event_id}.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_event(60)

In [ ]:
top = events.sort_values("rain_mm", ascending=False).head(5).index

for i in top:
    plot_event(i)

In [ ]:
print("Data range (IST):", data["time_ist"].min(), "to", data["time_ist"].max())
print("Total minutes of data:", len(data))
print("Days with data:", len(daily))
print("Wettest month:", monthly.idxmax(), "-", monthly.max(), "mm")

big = events.sort_values("rain_mm", ascending=False).iloc[0]
print("Biggest event:", big["start"], "to", big["end"], "-", big["rain_mm"], "mm")

peak_time = data.loc[data["intensity_1min"].idxmax(), "time_ist"]
print("Highest 1-min intensity:", data["intensity_1min"].max(), "mm/h at", peak_time)

In [ ]:
import shutil

os.makedirs("rain-rain-analysis/graphs", exist_ok=True)

ev = events.loc[60]
sample = data[(data["time_ist"] >= ev["start"]) & (data["time_ist"] <= ev["end"])]
sample[["time_ist", "intensity_1min", "rain_mm_min"]].to_csv("rain-rain-analysis/sample_event_2026-07-28.csv", index=False)

events.to_csv("rain-rain-analysis/events_summary.csv")

for i in top:
    shutil.copy(f"graphs/event_{i}.png", "rain-rain-analysis/graphs/")

print(os.listdir("rain-rain-analysis"))
print(os.listdir("rain-rain-analysis/graphs"))